In [1]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
from typing import List, Optional
import pandas as pd
import pyarrow as pya
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import glob
from itertools import chain
import numpy as np
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request

In [2]:
#df1 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump1.parquet')
#df2 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump2.parquet')
##df3 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump3.parquet')
#df4 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump4.parquet')
#df5 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump5.parquet')
df6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump6.parquet')
#df7 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategy\X-CANIDS\raw\dump7.parquet')

#dump1 = df1.to_pandas()
#dump2 = df2.to_pandas()
#dump3 = df3.to_pandas()
#dump4 = df4.to_pandas()
#dump5 = df5.to_pandas()
dump6 = df6.to_pandas()
#dump7 = df7.to_pandas()

In [3]:
def hex_to_bytes(h):
        """Convert hex/bytes/string to bytes."""
        if h is None or (isinstance(h, float) and np.isnan(h)): 
            return b""
        if isinstance(h, (bytes, bytearray)): 
            return bytes(h)
        s = str(h).strip().replace("0x","").replace(" ","")
        if s == "": 
            return b""
        if len(s) % 2 == 1: 
            s = "0"+s
        try: 
            return bytes.fromhex(s)
        except Exception: 
            return str(h).encode("utf-8", errors="ignore")


def hamming_distance(a: bytes, b: bytes) -> tuple:
    """
    Compute Hamming distance between two byte sequences.
    
    Key difference from Levenshtein:
    - Requires equal length (pads shorter with zeros)
    - Counts number of POSITIONS where bytes differ
    - Much faster: O(n) vs O(m*n)
    
    Args:
        a, b: Byte sequences to compare
        
    Returns:
        Number of differing bytes (0 to max_len)
    """
    len_mismatch = (len(a) != len(b))

    # Pad to same length
    max_len = max(len(a), len(b))
    a_padded = a + b'\x00' * (max_len - len(a))
    b_padded = b + b'\x00' * (max_len - len(b))
    
    # Count differences
    #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
    
    distance = 0
    for byte_a, byte_b in zip(a_padded, b_padded):
        distance += bin(byte_a ^ byte_b).count('1')
    return (distance, len_mismatch)
    

In [4]:
def compute_hamming_distances_training(dumps, out_csv, process_per_dump = True):
    
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    
    records = []
    prev_payload = {}  # {arbitration_id: previous_bytes}
    
    for dump_name, df in dumps:
        d = df.copy()
        
        # Ensure timestamp column
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        d = d.sort_values("timestamp", kind="mergesort")
        has_label = "label" in d.columns
        
        for _, row in d.iterrows():
            aid = row["arbitration_id"]
            ts = row["timestamp"]
            lab = int(row["label"]) if has_label and not pd.isna(row["label"]) else 0
            curr_bytes = hex_to_bytes(row["data"])
            
            # Get previous payload for this ID
            prev = prev_payload.get(aid)
            
            if prev is not None:
                # Compute Hamming distance
                dist, len_mismatch = hamming_distance(curr_bytes, prev)
                max_len = max(len(curr_bytes), len(prev))
                norm_dist = dist / (max_len*8) if max_len > 0 else 0.0
                
                records.append({
                    "dump": dump_name,
                    "timestamp": ts,
                    "arbitration_id": aid,
                    "payload_len": len(curr_bytes),
                    "ham_dist": dist,
                    "ham_norm": norm_dist,
                    "len_mismatch": len_mismatch,  
                    "label": lab
                })
            
            # Update previous payload
            prev_payload[aid] = curr_bytes
        
        if process_per_dump:
            prev_payload.clear()
    
    out_df = pd.DataFrame.from_records(records)
    out_df.to_csv(out_csv, index=False)
    print(f"✅ Saved: {out_csv} (rows={len(out_df):,})")
    
    return out_df
                
    
                    

In [6]:
benign_dumps = [
        ("dump6", dump6),
    ]

compute_hamming_distances_training(benign_dumps, "hamming_distances_normal_replay.csv")
range = pd.read_csv(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\hamming_distances_normal_replay.csv')
range.head()

✅ Saved: hamming_distances_normal_replay.csv (rows=4,279,845)


,dump,timestamp,arbitration_id,payload_len,ham_dist,ham_norm,len_mismatch,label
0,dump6,0 days 00:00:00.036195,356,4,4,0.125000,False,0
1,dump6,0 days 00:00:00.037291,273,8,3,0.046875,False,0
2,dump6,0 days 00:00:00.037528,274,8,3,0.046875,False,0
3,dump6,0 days 00:00:00.037766,275,8,3,0.046875,False,0
4,dump6,0 days 00:00:00.038002,128,8,6,0.093750,False,0


In [ ]:
def compute_hamming_distance_detection(aid, curr_data, payload_prev):# i have to pass the previous payload in the main
    prev = payload_prev.get(aid)
    if prev is not None:
        # Compute distance
        distance, _ = hamming_distance(curr_data, prev)
        max_distance = max(len(curr_data), len(prev)) * 8
        normalized_distance = distance / max_distance if max_distance > 0 else 0.0
        
        # Update previous
        payload_prev[aid] = curr_data
        
        return normalized_distance
    else:
        # First message for this ID
        payload_prev[aid] = curr_data
        return None

In [ ]:
def build_hamming_range_model(hamming_csv,model_csv):
                            
                        
        df = pd.read_csv(hamming_csv)
        # Group by ID and compute min/max
        ranges = (df.groupby('arbitration_id')['ham_dist']
                .agg(['min', 'max', 'count'])
                .reset_index())
        
        ranges.columns = ['arbitration_id', 'min_hamming', 'max_hamming', 'n_samples']
        
        # Compute range size
        ranges['range_size'] = ranges['max_hamming'] - ranges['min_hamming']
        
        # Normalized versions (for 8-byte payloads, max=8)
        ranges['min_norm'] = ranges['min_hamming'] / 8.0
        ranges['max_norm'] = ranges['max_hamming'] / 8.0
        ranges['range_norm'] = ranges['range_size'] / 8.0
        
        # Classify IDs (paper uses sigma=6 for byte-level Hamming)
        sigma = 6
        def classify(r):
            if r == 0:
                return 'NoRange'
            elif r <= sigma:
                return 'SmallRange'
            else:
                return 'MidRange'
            
        
        ranges['class'] = ranges['range_size'].apply(classify)
        
        ranges = ranges.sort_values('range_size')
        
        # Save
        Path(model_csv).parent.mkdir(parents=True, exist_ok=True)
        ranges.to_csv(model_csv, index=False)
        

In [ ]:
def detection(attack_csv, ranges_csv):
    
    test = pd.read_csv(attack_csv)
    ranges = pd.read_csv(ranges_csv)
    # TO DO se arriva un pacchetto singolo io devo essere in grado di dire se è anomalo o no, come sopra avrò in qualche modo tenuto con un dictionary l'id e il payload del pacchetto precedente e avrò la loro distanza. qui devo semplicemente vedere se sono dentro o fuori il range.
    

In [ ]:
replay_file_testing = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\raw\dump6-repl-0-120.parquet"

